# Herramienta 02 - Clasificación de conducción distractiva

Este notebook funciona como una herramienta de clasificación visual: prepara una muestra de imágenes, extrae características, entrena un modelo, evalúa errores y permite clasificar una imagen nueva.


## 1. Configuración
Se usan librerías livianas para que el notebook corra rápido en Colab. La herramienta descarga una muestra pequeña versionada en el repositorio.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
BASE_URL = 'https://raw.githubusercontent.com/AndresGuido9820/sistema-transporte-inteligente/main'
LOCAL_DIR = Path('driver_sample')
LOCAL_DIR.mkdir(exist_ok=True)


## 2. Carga de metadatos e imágenes
El CSV contiene ruta de imagen y etiqueta. Se descargan las imágenes necesarias para el entrenamiento local en Colab.


In [ ]:
metadata_url = f'{BASE_URL}/data/processed/driver_images.csv'
metadata = pd.read_csv(metadata_url)
metadata = metadata.groupby('label', group_keys=False).head(30).reset_index(drop=True)

local_paths = []
for i, row in metadata.iterrows():
    remote_path = row['image_path'].replace('data/processed/', '')
    url = f'{BASE_URL}/data/processed/{remote_path}'
    local_path = LOCAL_DIR / f'{i:03d}_{Path(remote_path).name}'
    if not local_path.exists():
        urlretrieve(url, local_path)
    local_paths.append(str(local_path))

metadata['local_path'] = local_paths
print('Imágenes cargadas:', metadata.shape[0])
display(metadata['label'].value_counts().rename_axis('clase').reset_index(name='imagenes'))


## 3. Exploración visual
Se muestran ejemplos por clase para comprobar que el problema es visualmente razonable y que las etiquetas tienen sentido.


In [ ]:
clases = sorted(metadata['label'].unique())
plt.figure(figsize=(12, 7))
plot_index = 1
for clase in clases:
    ejemplos = metadata[metadata['label'] == clase].head(3)
    for _, row in ejemplos.iterrows():
        plt.subplot(len(clases), 3, plot_index)
        plt.imshow(Image.open(row['local_path']).convert('RGB'))
        plt.title(clase, fontsize=9)
        plt.axis('off')
        plot_index += 1
plt.tight_layout()
plt.show()


## 4. Extracción de características
Para mantener la herramienta rápida, se usan características de color, contraste y oscuridad. En un despliegue real se puede reemplazar esta parte por MobileNet, ResNet o EfficientNet.


In [ ]:
def extraer_caracteristicas(path):
    imagen = Image.open(path).convert('RGB').resize((64, 64))
    arr = np.asarray(imagen, dtype=float) / 255.0
    medias = arr.mean(axis=(0, 1))
    desvios = arr.std(axis=(0, 1))
    brillo = arr.mean(axis=2)
    dark_ratio = (brillo < 0.20).mean()
    bright_ratio = (brillo > 0.80).mean()
    return np.concatenate([medias, desvios, [dark_ratio, bright_ratio]])

X = np.vstack([extraer_caracteristicas(path) for path in metadata['local_path']])
y = metadata['label'].values
print('Matriz de entrenamiento:', X.shape)


## 5. Entrenamiento y métricas
Se entrena un clasificador multiclase y se reportan accuracy, precisión, recall y F1 por clase.


In [ ]:
X_train, X_test, y_train, y_test, path_train, path_test = train_test_split(
    X, y, metadata['local_path'].values,
    test_size=0.25,
    random_state=SEED,
    stratify=y,
)

modelo = RandomForestClassifier(n_estimators=160, random_state=SEED, class_weight='balanced')
modelo.fit(X_train, y_train)
pred = modelo.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, pred), 3))
print(classification_report(y_test, pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, pred, xticks_rotation=45, cmap='Blues')
plt.title('Matriz de confusión')
plt.tight_layout()
plt.show()


## 6. Aciertos y errores
Esta sección ayuda a explicar en el reporte qué clases se confunden y por qué se necesitan más datos o una CNN para mejorar.


In [ ]:
revision = pd.DataFrame({'path': path_test, 'real': y_test, 'prediccion': pred})
revision['correcta'] = revision['real'] == revision['prediccion']
display(revision.groupby(['real', 'prediccion']).size().reset_index(name='casos').sort_values('casos', ascending=False).head(10))

muestras = pd.concat([revision[revision['correcta']].head(3), revision[~revision['correcta']].head(3)])
plt.figure(figsize=(12, 4))
for i, (_, row) in enumerate(muestras.iterrows(), start=1):
    plt.subplot(1, len(muestras), i)
    plt.imshow(Image.open(row['path']).convert('RGB'))
    plt.title(f"R: {row['real']}\nP: {row['prediccion']}", fontsize=8)
    plt.axis('off')
plt.tight_layout()
plt.show()


## 7. Herramienta de clasificación
Cambie `imagen_a_clasificar` por la ruta de una imagen subida a Colab. Si se deja vacío, se usa una imagen de prueba.


In [ ]:
#@title Parámetros de la herramienta
imagen_a_clasificar = '' #@param {type:'string'}

if not imagen_a_clasificar:
    imagen_a_clasificar = path_test[0]

features_img = extraer_caracteristicas(imagen_a_clasificar).reshape(1, -1)
clase = modelo.predict(features_img)[0]
probas = pd.Series(modelo.predict_proba(features_img)[0], index=modelo.classes_).sort_values(ascending=False)

plt.figure(figsize=(5, 4))
plt.imshow(Image.open(imagen_a_clasificar).convert('RGB'))
plt.title(f'Clase predicha: {clase}')
plt.axis('off')
plt.show()

display(probas.rename('probabilidad').reset_index().rename(columns={'index': 'clase'}))
probas.plot(kind='bar', figsize=(8, 3), title='Probabilidad por clase')
plt.ylabel('Probabilidad')
plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.25)
plt.show()


## 8. Medidas preventivas sugeridas
Las clases asociadas al uso del teléfono y otras actividades deben activar campañas de capacitación, alertas preventivas y revisión humana. El modelo no debe usarse como sanción automática.
